# Chronos-2 Forecasting — DIMER `TASK-INFERENCE`

**Notebook specification:** DIMER Notebook Specification 1.0

Chronos-2 supplies the pretrained zero-shot forecasting model. This repository supplies DIMER configuration and validation, immutable model pinning/digest checks, normalized outputs, chronological evaluation, baselines, and provenance. **No training or fine-tuning occurs.**

By the end you can: bootstrap the locked runtime; validate bundled or BYOD CSV data; create a leakage-safe chronological holdout; verify the pinned `amazon/chronos-2` checkpoint; run univariate and **multi-target (Mode C)** forecasts; interpret median/quantile outputs and tutorial metrics; optionally run known-future covariates (Mode D); and export forecasts plus provenance.

Out of scope: classification, anomaly detection, imputation, embeddings, training/fine-tuning, calibrated prediction intervals, and production-fitness claims.

References: [README](../README.md) · [MODEL_CARD](../MODEL_CARD.md) · [sample dataset card](../examples/sample-data/DATASET_CARD.md) · [upstream Chronos](https://github.com/amazon-science/chronos-forecasting) · [pinned model](https://huggingface.co/amazon/chronos-2)

**Prerequisites:** Python 3.12; CPU is the default path; first model load may download ~478 MB. BYOD is read locally in the notebook runtime and is not sent to an external inference service. Do not upload confidential, restricted, personal, or otherwise sensitive data to hosted notebooks unless authorized.


## 1. Bootstrap the locked runtime

The notebook installs `requirements.lock.txt` with hashes, then installs this repository without resolving a second dependency graph. Automation can set `DIMER_TUTORIAL_REF` to an immutable branch/commit. If core packages were already imported and the install replaces them, the cell fails with a restart instruction rather than mixing stale in-memory modules.


In [ ]:
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kurtvalcorza/chronos-2-forecasting-pipeline.git"
REPO_NAME = "chronos-2-forecasting-pipeline"
REPO_REF = os.environ.get("DIMER_TUTORIAL_REF", "main")
SKIP_INSTALL = os.environ.get("DIMER_NOTEBOOK_CI_PREINSTALLED") == "1"

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(["git", "clone", "--filter=blob:none", "-q", REPO_URL, str(checkout)], check=True)
    if REPO_REF != "main":
        subprocess.run(["git", "-C", str(checkout), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "-C", str(checkout), "checkout", "-q", "main"], check=True)
        subprocess.run(["git", "-C", str(checkout), "pull", "--ff-only", "-q", "origin", "main"], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    tracked = {"numpy": "numpy", "pandas": "pandas", "torch": "torch"}
    loaded = {m: getattr(sys.modules[m], "__version__", None) for m in tracked.values() if m in sys.modules}
    if shutil.which("uv"):
        subprocess.run(["uv", "pip", "install", "--python", sys.executable, "--require-hashes", "-r", str(ROOT / "requirements.lock.txt")], check=True)
        subprocess.run(["uv", "pip", "install", "--python", sys.executable, "--no-deps", "-e", str(ROOT)], check=True)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--require-hashes", "-r", str(ROOT / "requirements.lock.txt")], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(ROOT)], check=True)
    stale = [f"{m}: loaded={loaded[m]}, installed={importlib.metadata.version(d)}" for d, m in tracked.items() if m in loaded and loaded[m] not in (None, importlib.metadata.version(d))]
    if stale:
        raise RuntimeError("Core dependencies changed while older modules were loaded: " + "; ".join(stale) + ". Restart the notebook runtime, then rerun from the top.")

print("repository:", ROOT)
print("repository ref:", REPO_REF)


## 2. Runtime identity and operational ceilings

The pinned model exposes an 8,192-step context and native 1,024-step horizon. DIMER request guards cap one request at 1,000 series IDs, 64 targets, 64 covariates, 5,000,000 rows, 8,192 context steps, and 4,096 forecast steps. Horizons above 1,024 require explicit autoregressive unrolling in the pipeline.

The tutorial uses CPU, a 12-step horizon, deterministic synthetic data, and a deterministic chronological split—no random split or training initialization. Floating-point details can vary across runtime/hardware builds and latency is run-dependent.


In [ ]:
import platform
from chronos2_pipeline import ResourceLimits, runtime_versions

versions = runtime_versions()
print("Python:", platform.python_version())
print("chronos-forecasting:", versions.get("chronos-forecasting"))
print("torch:", versions.get("torch"))
print("transformers:", versions.get("transformers"))
print("DIMER request limits:", ResourceLimits())
print("Pinned-model context limit: 8192")
print("Pinned-model native prediction length: 1024")


## 3. Bundled sample or BYOD

The bundled sample is deterministic synthetic teaching data, not benchmark evidence; see the dataset card. BYOD requires unique UTF-8 CSV headers plus `series_id`, `timestamp`, and `target`. Timestamps must be parseable, regular, and contiguous per series; targets must be finite numeric. Duplicate headers are rejected **before pandas can rename them**. Additional numeric covariates are permitted by the pipeline.

Set `USE_BYOD=True` in Colab, or set `DIMER_BYOD_PATH` in automation.


In [ ]:
import csv
import hashlib
import io
import json
from collections import Counter
import pandas as pd
from chronos2_pipeline import ForecastConfig, chronological_holdout, evaluate_forecast, forecast, last_value_baseline, load_pinned_model, seasonal_naive_baseline

USE_BYOD = False  # @param {type:"boolean"}
PREDICTION_LENGTH = 12  # @param {type:"integer"}
BYOD_PATH = os.environ.get("DIMER_BYOD_PATH")

def read_checked_csv(payload: bytes) -> pd.DataFrame:
    rows = csv.reader(io.StringIO(payload.decode("utf-8-sig")))
    try:
        header = next(rows)
    except StopIteration as exc:
        raise ValueError("CSV is empty.") from exc
    duplicates = sorted(k for k, v in Counter(header).items() if v > 1)
    if duplicates:
        raise ValueError(f"Duplicate CSV header(s) are ambiguous: {duplicates}")
    return pd.read_csv(io.BytesIO(payload))

if BYOD_PATH:
    payload = Path(BYOD_PATH).read_bytes()
    input_source = f"BYOD path: {BYOD_PATH}"
elif USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV.")
    name, payload = next(iter(uploaded.items()))
    input_source = f"BYOD upload: {name}"
else:
    sample_path = ROOT / "examples" / "sample-data" / "chronos_univariate.csv"
    payload = sample_path.read_bytes()
    expected = (sample_path.parent / "SHA256SUMS").read_text().split()[0]
    observed = hashlib.sha256(payload).hexdigest()
    if observed != expected:
        raise ValueError(f"Bundled sample digest mismatch: {observed}")
    input_source = f"bundled synthetic sample: {sample_path}"

frame = read_checked_csv(payload)
input_sha256 = hashlib.sha256(payload).hexdigest()
print(input_source)
print("input sha256:", input_sha256)
print(frame.head())


## 4. Leakage-safe chronological holdout and pinned model

The final `PREDICTION_LENGTH` timestamps of each series are held out as truth; future target values never enter model context. `load_pinned_model()` accepts only the approved immutable Chronos-2 revision, verifies the resolved snapshot plus `model.safetensors` byte size/SHA-256 and `config.json` SHA-256, and refuses pickle-style weights.


In [ ]:
config = ForecastConfig(target="target", prediction_length=PREDICTION_LENGTH, quantile_levels=[0.1, 0.5, 0.9], device="cpu")
split = chronological_holdout(frame, config)
for sid in split.history[config.id_column].drop_duplicates():
    h = split.history[split.history[config.id_column] == sid]
    t = split.truth[split.truth[config.id_column] == sid]
    assert h[config.timestamp_column].max() < t[config.timestamp_column].min()

model = load_pinned_model(device="cpu")
print("model:", model.identity.model_id)
print("revision:", model.identity.revision)
print("weights sha256:", model.identity.weights_sha256)
print("config sha256:", model.identity.config_sha256)
print("device/dtype:", model.device, model.dtype)


## 5. Forecast and evaluate

Normalized output fields are `series_id`, `timestamp`, `target_name`, `prediction`, and requested quantiles such as `q0.1`, `q0.5`, `q0.9`. `prediction` is the **median (`q0.5`)**, not a mean. Model quantiles summarize the predictive distribution; they are **not guaranteed frequentist confidence intervals** and are not assumed calibrated on a new domain.

Metrics use the same chronological holdout: **MAE** is mean absolute error; **RMSE** weights larger misses more; **pinball loss** evaluates a quantile asymmetrically; **empirical interval coverage** is the fraction of held-out truths inside the requested outer quantiles. These are tutorial/sanity metrics, not benchmark evidence.

The seasonal-naive comparator runs only for uniformly hourly data when **every series has at least 24 post-holdout history rows**. Short but otherwise valid hourly BYOD therefore skips this optional comparator instead of failing.


In [ ]:
result = forecast(split.history, config, model)
evaluation = evaluate_forecast(result.forecast, split.truth, config)
last_value_evaluation = evaluate_forecast(last_value_baseline(split.history, split.truth, config), split.truth, config)

ordered = split.history.sort_values([config.id_column, config.timestamp_column])
steps = ordered.groupby(config.id_column)[config.timestamp_column].diff().dropna()
minimum_history = int(ordered.groupby(config.id_column, sort=False).size().min())
can_use_daily_seasonal = (not steps.empty) and (steps == pd.Timedelta(hours=1)).all() and minimum_history >= 24

seasonal_evaluation = None
if can_use_daily_seasonal:
    seasonal = seasonal_naive_baseline(split.history, split.truth, config, season_length=24)
    seasonal_evaluation = evaluate_forecast(seasonal, split.truth, config)

print(result.forecast.head())
print("Chronos-2:", evaluation.aggregate)
print("Last-value:", last_value_evaluation.aggregate)
print("Seasonal-naive:", None if seasonal_evaluation is None else seasonal_evaluation.aggregate)
print("Quantile metrics:")
print(evaluation.quantiles)
print("Per-series metrics:")
print(evaluation.per_series)


## 6. Primary Mode C — multi-target forecasting

Multi-target forecasting is a primary user-facing capability, so this path runs by default. For interface demonstration only, `target_aux` is deterministically derived from `target`; it is **not an independent benchmark variable**. The assertion proves the normalized output preserves both target names.


In [ ]:
mode_c_frame = frame.copy()
mode_c_frame["target_aux"] = 0.5 * pd.to_numeric(mode_c_frame["target"]) + 10.0
mode_c_config = ForecastConfig(target=["target", "target_aux"], prediction_length=PREDICTION_LENGTH, quantile_levels=[0.1, 0.5, 0.9], device="cpu")
mode_c_split = chronological_holdout(mode_c_frame, mode_c_config)
mode_c_result = forecast(mode_c_split.history, mode_c_config, model)
assert set(mode_c_result.forecast["target_name"].unique()) == {"target", "target_aux"}
print("Mode C target names:", sorted(mode_c_result.forecast["target_name"].unique()))


## 7. Export machine-readable evidence

Exports contain the normalized univariate forecast, Mode C multi-target forecast, per-series evaluation, aggregate/quantile metrics, and provenance. A BYOD SHA-256 is metadata derived from uploaded bytes and should be retained/disclosed under your data-governance rules.


In [ ]:
OUTPUT_DIR = Path(os.environ.get("DIMER_OUTPUT_DIR", ROOT / "outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

result.forecast.to_csv(OUTPUT_DIR / "chronos_forecast.csv", index=False)
mode_c_result.forecast.to_csv(OUTPUT_DIR / "chronos_multitarget_forecast.csv", index=False)
evaluation.per_series.to_csv(OUTPUT_DIR / "chronos_evaluation_per_series.csv", index=False)

provenance = {**result.provenance, "tutorial": {"notebook_profile": "TASK-INFERENCE", "notebook_spec": "1.0", "input_source": input_source, "input_sha256": input_sha256, "primary_capability_checks": ["univariate", "multi-target"]}}
(OUTPUT_DIR / "chronos_provenance.json").write_text(json.dumps(provenance, indent=2, default=str), encoding="utf-8")

metrics = {"evidence_scope": "tutorial/sanity; not benchmark or production-fitness evidence", "estimation_procedure": "single chronological tail holdout", "chronos2": evaluation.aggregate, "last_value": last_value_evaluation.aggregate, "seasonal_naive_24": None if seasonal_evaluation is None else seasonal_evaluation.aggregate, "quantiles": evaluation.quantiles.to_dict("records")}
(OUTPUT_DIR / "chronos_evaluation.json").write_text(json.dumps(metrics, indent=2, default=str), encoding="utf-8")

print("wrote:", sorted(p.name for p in OUTPUT_DIR.iterdir()))


## Optional: Mode D known-future covariates

Enable `RUN_COVARIATE_DEMO` or set `DIMER_RUN_COVARIATE_DEMO=1`. The deterministic future table contains only `temperature` and `holiday`, never future `demand`, and is built in memory so tracked files are not rewritten.


In [ ]:
RUN_COVARIATE_DEMO = False  # @param {type:"boolean"}
if os.environ.get("DIMER_RUN_COVARIATE_DEMO") == "1":
    RUN_COVARIATE_DEMO = True

if RUN_COVARIATE_DEMO:
    import importlib.util
    generator_path = ROOT / "examples" / "sample-data" / "generate_samples.py"
    spec = importlib.util.spec_from_file_location("chronos_sample_generator", generator_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not load {generator_path}")
    generator = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(generator)
    samples = generator.build_samples()
    cov_history = samples["chronos_covariates_history.csv"]
    cov_future = samples["chronos_covariates_future.csv"]
    assert "demand" not in cov_future.columns
    cov_config = ForecastConfig(target="demand", prediction_length=24, quantile_levels=[0.1, 0.5, 0.9], device="cpu")
    cov_result = forecast(cov_history, cov_config, model, cov_future)
    print("known-future covariates:", cov_result.inference["known_future_covariate_names"])
else:
    print("Mode D demo skipped.")


## Interpretation and limits

A successful default run **proves** that the recorded runtime can install the locked repository, verify/load the pinned model, preserve the chronological evaluation boundary, execute the public API for univariate and primary multi-target forecasting, compute the documented tutorial metrics/baselines, and export machine-readable forecasts/provenance.

It **does not prove** accuracy, calibration, robustness, fairness, safety, or production readiness for your domain. The synthetic sample is intentionally simple; its metrics are not benchmark evidence. BYOD requires representative multi-origin backtesting, leakage controls, domain baselines, calibration checks, and operational review.

Current limits include fixed-width regular frequencies; monthly/quarterly/yearly/business-day, irregular, and gappy calendars are outside this path.
